# 03 — Text Translation

## Purpose
Classifies each parsed document page as `ENGLISH`, `BILINGUAL`, or `NON_ENGLISH`
and translates pages that require it before field extraction runs. This ensures
the extraction model always receives English text regardless of the source
document language.

## What this notebook does

### Language Classification
Each page in `DOCUMENTS_PAGES` is first passed through a two-stage classifier:

**Stage 1 — Rule-based pre-filter** runs before any AI call. It counts
non-Latin script characters (CJK, Cyrillic, Arabic) as a proportion of total
page characters. Pages where non-Latin script exceeds 5% of content AND fewer
than 30 Latin words are present are classified as `NON_ENGLISH` immediately
without calling the model. This avoids unnecessary AI token spend on clearly
non-English pages such as pure Russian or pure Chinese documents.

**Stage 2 — AI_COMPLETE classification** handles all remaining pages using
`claude-haiku-4-5` — the cheapest available model, sufficient for a binary
classification task. The prompt instructs the model to:
- Ignore proper nouns and address terms in Latin script (vessel names, port
  names, company names, Indonesian administrative terms like `Desa`,
  `Kecamatan`)
- Focus only on descriptive field values (species names, product descriptions,
  processing terms, units of measure)
- Return a structured JSON response with a `label` and a `reason` for
  human-readable audit

**Classification labels and downstream behavior:**

| Label | Meaning | Action |
|---|---|---|
| `ENGLISH` | All field values in English or language-neutral | No translation — extraction reads `PAGE_CONTENT` directly |
| `BILINGUAL` | Every field value present in both English and a non-English language | No translation — English equivalent already present on page |
| `NON_ENGLISH` | Field values in a non-English language with no English equivalent | Translated before extraction |

### Translation
Only `NON_ENGLISH` pages are translated. `BILINGUAL` pages are skipped because
the English equivalent is already present — the extraction model reads those
directly from `PAGE_CONTENT`.

Translation uses `AI_COMPLETE` with `claude-sonnet-5` via a parallel SQL
pattern — `REPLACE()` injects each page's content into the prompt template at
SQL execution time, and Snowflake runs `AI_COMPLETE` across all rows
simultaneously. This is the same parallel pattern used in classification and
extraction — no Python threading needed.

The translation prompt instructs the model to:
- Translate all non-English text including non-Latin script administrative
  terms 
- Preserve Latin-script proper nouns exactly as-is (vessel names, port names,
  company names, `Desa`, `Kecamatan`)
- Preserve scientific names, HS codes, product codes, dates, and numbers
- Maintain the exact document structure — table pipes, column layout, section
  numbers, and blank rows must be unchanged

## Outputs

| Table | What is written |
|---|---|
| `PROCESSING.DOCUMENTS_PAGES` | `LANGUAGE_RESULT`, `LANGUAGE_CLASSIFICATION_REASON`, `LANGUAGE_CHECKED_AT` updated per page |
| `PROCESSING.DOCUMENTS_PAGES` | `PAGE_CONTENT_TRANSLATED` written for `NON_ENGLISH` pages only |
| `AUDIT.LLM_USAGE` | Token consumption per page for both classification and translation steps |

## Key design decisions
- **Three-label classification** — `BILINGUAL` is a distinct label from
  `NON_ENGLISH` because bilingual documents (Chinese health certificates,
  Russian catch certificates with side-by-side English) already contain English
  field values and do not need translation. Treating them as `NON_ENGLISH`
  would waste tokens and risk overwriting correct English values with a
  less-accurate translation
- **Rule-based pre-filter before AI** — character counting is deterministic
  and free. Pages with dominant non-Latin script are classified without any
  model call, reducing token spend on the most obvious cases
- **`AI_COMPLETE` over `AI_CLASSIFY`** — `AI_COMPLETE` returns a structured
  JSON response with a `reason` field that explains each classification
  decision. This makes misclassifications debuggable without re-running the
  model. `AI_CLASSIFY` was found to be inconsistent on pages with Icelandic
  special characters in vessel names and Indonesian address fragments
- **`claude-haiku-4-5` for classification, `claude-sonnet-5` for translation**
  — classification is a simple structured output task that haiku handles
  reliably at lower cost. Translation requires higher quality output to
  preserve document structure and translate domain-specific seafood terminology
  accurately

In [ ]:
import json
import re
import pandas as pd
import pytz
import uuid
from datetime import datetime, timezone
from snowflake.snowpark.context import get_active_session

DB                = 'PERMAFROST_POC'
PROCESSING_SCHEMA = 'PROCESSING'
AUDIT_SCHEMA = 'AUDIT'
s = get_active_session() 

def info(msg):    print(f"INFO:    {msg}")
def warning(msg): print(f"WARNING: {msg}")
def error(msg):   print(f"ERROR:   {msg}")

def now_ast(): # could be replace with any time zone later
    tz = pytz.timezone('America/Halifax')
    return datetime.now(tz).isoformat()


In [ ]:
def classify_language_rule_based(text):
    """
    Rule-based pre-filter — only classifies pages where the answer is certain.
    Returns None for ambiguous cases which get passed to AI_COMPLETE.

    Only handles one clear case:
    - Pages dominated by non-Latin script with very
      little English present → NON_ENGLISH for certain

    Everything else returns None → AI_COMPLETE decides.
    """
    if not text or not text.strip():
        return None

    total_chars = len(text.strip())

    # Count non-Latin script characters
    cjk_chars = sum(
        1 for c in text
        if '\u4e00' <= c <= '\u9fff'   # CJK Unified Ideographs
        or '\u3400' <= c <= '\u4dbf'   # CJK Extension A
        or '\uac00' <= c <= '\ud7af'   # Korean
        or '\u3040' <= c <= '\u309f'   # Hiragana
        or '\u30a0' <= c <= '\u30ff'   # Katakana
    )

    cyrillic_chars = sum(
        1 for c in text
        if '\u0400' <= c <= '\u04ff'
    )

    arabic_chars = sum(
        1 for c in text
        if '\u0600' <= c <= '\u06ff'
    )

    non_latin_chars = cjk_chars + cyrillic_chars + arabic_chars
    non_latin_ratio = non_latin_chars / total_chars

    # Only classify NON_ENGLISH when confident
    # high non-Latin density AND very little Latin text present
    if non_latin_ratio > 0.05:
        latin_words = len(re.findall(r'\b[a-zA-Z]{3,}\b', text))

        # If non-Latin dominates and Latin words are few → definitely NON_ENGLISH
        if latin_words < 30:
            return 'NON_ENGLISH'
        else:
            return None   # Ambiguous — could be bilingual

    return None

In [ ]:
LANGUAGE_CHECK_PROMPT = """You are classifying a page from a trade or fisheries document to determine if translation is needed.

ALWAYS IGNORE when classifying (these never need translation):
- Vessel names, port names, company names, farm names, person names in Latin script
- Address administrative terms in Latin script (Desa, Kecamatan, Kabupaten, and equivalents in other languages)
- Romanized place names (DALIAN, LIAONING, GUANGDONG, EAST JAVA, Þorlákshöfn, Vladivostok)
- Scientific Latin names, HS codes, product codes, certificate numbers, dates, numbers, country codes

ENGLISH: All descriptive field values (species, product, processing method, units) are in English or language-neutral. Any non-Latin script is limited to a company seal/stamp/logo. The only non-English elements are proper nouns from the IGNORE list.

BILINGUAL: Every key field value appears in BOTH English AND a non-English language on the same page. The page provides its own English translation — no external translation needed. Example: Chinese health certificate or Russian catch certificate with side-by-side English and non-English labels and values.

NON_ENGLISH: Descriptive field values appear ONLY in a non-English language with no English equivalent on the same page. English column headers do NOT count as equivalents for the values beneath them. Example: Icelandic worksheet with Þorskur, kassi, IQF-BITAR where English headers exist but values have no English equivalent.

CRITICAL: First identify proper nouns and ignore them. Then ask: Are the remaining descriptive VALUES in English? If yes → ENGLISH. If no, does the page provide English equivalents? If yes → BILINGUAL. If no → NON_ENGLISH.

Document page:
{page_text}

Return a JSON object with exactly this structure and nothing else:
{{"label": "ENGLISH" or "BILINGUAL" or "NON_ENGLISH", "reason": "one sentence explaining the key signal that determined the classification"}}"""

# Language classification using AI_COMPLETE

rows = s.sql(f"""
    SELECT
        DOC_ID,
        PAGE_INDEX,
        PAGE_NUMBER,
        PAGE_CONTENT
    FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_PAGES
    WHERE PAGE_CONTENT      IS NOT NULL
      AND TRIM(PAGE_CONTENT) <> ''
      AND LANGUAGE_RESULT   IS NULL
""").collect()

info(f"Loaded {len(rows)} page(s) for language classification")

# ── Rule-based pre-filter 
pre_classified = []
needs_ai       = []

for row in rows:
    rule_result = classify_language_rule_based(row['PAGE_CONTENT'])
    if rule_result:
        pre_classified.append({
            'DOC_ID':              row['DOC_ID'],
            'PAGE_INDEX':          row['PAGE_INDEX'],
            'PAGE_NUMBER':         row['PAGE_NUMBER'],
            'PAGE_CONTENT':        row['PAGE_CONTENT'],
            'LANGUAGE_RESULT':     rule_result,
            'LANGUAGE_CLASSIFICATION_REASON':  'Rule-based: non-Latin script density threshold',
            'ESTIMATED_INPUT_TOKENS': 0,
        })
    else:
        needs_ai.append(row)

info(f"Pre-classified by rules : {len(pre_classified)} pages")
info(f"Needs AI_COMPLETE       : {len(needs_ai)} pages")

In [ ]:
# AI_COMPLETE for remaining pages
ai_classified = []

if needs_ai:
    values_list = ', '.join(
        f"('{r['DOC_ID']}', {r['PAGE_INDEX']})"
        for r in needs_ai
    )

    try:
        results = s.sql(f"""
            SELECT
                p.DOC_ID,
                p.PAGE_INDEX,
                p.PAGE_NUMBER,
                p.PAGE_CONTENT,
                LENGTH(p.PAGE_CONTENT)  AS CONTENT_CHARS,
                AI_COMPLETE(
                    'claude-haiku-4-5',
                    REPLACE(?, '{{page_text}}', p.PAGE_CONTENT)
                ) AS CLASSIFICATION
            FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_PAGES p
            JOIN (
                SELECT DISTINCT DOC_ID, PAGE_INDEX
                FROM (VALUES {values_list}) AS t(DOC_ID, PAGE_INDEX)
            ) AS to_classify
                ON  p.DOC_ID     = to_classify.DOC_ID
                AND p.PAGE_INDEX = to_classify.PAGE_INDEX
            WHERE p.PAGE_CONTENT IS NOT NULL
              AND TRIM(p.PAGE_CONTENT) <> ''
        """, params=[LANGUAGE_CHECK_PROMPT]).collect()

        info(f"  AI_COMPLETE returned {len(results)} result(s)")

    except Exception as e:
        error(f"  SQL classification failed: {e}")
        results = []

    for row in results:
        raw = row['CLASSIFICATION']
        doc_id     = row['DOC_ID']
        page_index = row['PAGE_INDEX']

        try:
            cleaned = raw.strip()
        
            # If the whole response is a JSON string (double-quoted), unwrap it first
            if cleaned.startswith('"'):
                cleaned = json.loads(cleaned)  # returns the inner string
        
            # Strip markdown code fences
            if cleaned.strip().startswith('```'):
                parts = cleaned.strip().split('```')
                cleaned = parts[1].strip()
                if cleaned.lower().startswith('json'):
                    cleaned = cleaned[4:].strip()
        
            # Parse the actual JSON object
            parsed = json.loads(cleaned)
            if isinstance(parsed, str):
                parsed = json.loads(parsed)
        
            if not isinstance(parsed, dict):
                raise ValueError(f"Expected dict after parsing, got {type(parsed)}")
        
            label  = parsed.get('label', '').upper().strip()
            reason = parsed.get('reason', '')
        
            if label not in ('ENGLISH', 'BILINGUAL', 'NON_ENGLISH'):
                raise ValueError(f"Unexpected label: {label}")
        
            ai_classified.append({
                'DOC_ID':                 doc_id,
                'PAGE_INDEX':             page_index,
                'PAGE_NUMBER':            row['PAGE_NUMBER'],
                'PAGE_CONTENT':           row['PAGE_CONTENT'],
                'LANGUAGE_RESULT':        label,
                'LANGUAGE_CLASSIFICATION_REASON': reason,
                'ESTIMATED_INPUT_TOKENS': row['CONTENT_CHARS'] // 4,
            })
        
            info(f"  [{label}] DOC_ID: {doc_id} PAGE: {page_index} — {reason}")
        
        except Exception as e:
            error(f"  [FAIL] DOC_ID: {doc_id} PAGE: {page_index}: {e}")
            error(f"  RAW: {str(raw)[:200]}")

# Combine all classified pages
language_check = pre_classified + ai_classified

info(f"\nTotal classified: {len(language_check)} pages "
     f"({len(pre_classified)} by rules, {len(ai_classified)} by AI)")

In [ ]:
# UPDATE DOCUMENTS_PAGES & INSERT LLM_USAGE 
if language_check:
    s.write_pandas(
        pd.DataFrame([{
            'DOC_ID':              row['DOC_ID'],
            'PAGE_INDEX':          row['PAGE_INDEX'],
            'LANGUAGE_RESULT':     row['LANGUAGE_RESULT'],
            'LANGUAGE_CLASSIFICATION_REASON':  row.get('LANGUAGE_CLASSIFICATION_REASON', ''), 
            'LANGUAGE_CHECKED_AT': now_ast(),
            'TOKENS_IN':           row.get('ESTIMATED_INPUT_TOKENS', 0),
            'TOKENS_OUT':          2,
        } for row in language_check]),
        table_name='LANGUAGE_STAGING',
        database=DB, schema=PROCESSING_SCHEMA,
        overwrite=True,
        auto_create_table=True,
    )

    s.sql(f"""
        UPDATE {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_PAGES p
        SET
            p.LANGUAGE_RESULT     = t.LANGUAGE_RESULT,
            p.LANGUAGE_CLASSIFICATION_REASON   = t.LANGUAGE_CLASSIFICATION_REASON, 
            p.LANGUAGE_CHECKED_AT = t.LANGUAGE_CHECKED_AT
        FROM {DB}.{PROCESSING_SCHEMA}.LANGUAGE_STAGING t
        WHERE p.DOC_ID         = t.DOC_ID
          AND p.PAGE_INDEX     = t.PAGE_INDEX
          AND p.LANGUAGE_RESULT IS NULL
    """).collect()

    info(f"Updated {len(language_check)} page(s) in DOCUMENTS_PAGES")

    s.sql(f"""
        INSERT INTO {DB}.{AUDIT_SCHEMA}.LLM_USAGE
            (DOC_ID, CHILD_DOC_ID, PIPELINE_STEP, MODEL_NAME, TOKENS_IN, TOKENS_OUT, CALLED_AT)
        SELECT
            DOC_ID,
            NULL,
            'LANGUAGE_CHECK',
            'claude-haiku-4-5',
            TOKENS_IN,
            TOKENS_OUT,
            LANGUAGE_CHECKED_AT
        FROM {DB}.{PROCESSING_SCHEMA}.LANGUAGE_STAGING
        WHERE TOKENS_IN > 0   -- exclude rule-based rows from LLM_USAGE
    """).collect()

    info(f"Wrote {len(ai_classified)} LLM_USAGE row(s)")

    s.sql(f"""
        DROP TABLE IF EXISTS {DB}.{PROCESSING_SCHEMA}.LANGUAGE_STAGING
    """).collect()

#  Summary
print(f"\n Language classification summary")
s.sql(f"""
    SELECT
        LANGUAGE_RESULT,
        COUNT(DISTINCT DOC_ID)  AS DOCS,
        COUNT(*)                AS PAGES
    FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_PAGES
    WHERE LANGUAGE_RESULT IS NOT NULL
    GROUP BY LANGUAGE_RESULT
    ORDER BY LANGUAGE_RESULT
""").show()

In [ ]:
s.sql(f"""
    UPDATE {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_PAGES
    SET LANGUAGE_checked_at= NULL
""").collect()

In [ ]:
TRANSLATE_MODEL = 'claude-sonnet-5'
TRANSLATE_PROMPT = """You are translating a trade or fisheries document page to English.

RULES:
1. Translate all non-English text to English, including: field values, product descriptions, species names, processing terms, units of measure, remarks, and all non-Latin script text
2. Translate non-Latin administrative terms (District, City, Street, Province, etc.)
3. KEEP AS-IS: Latin-script proper nouns (vessel names, port names, company names, address terms like Desa, Kecamatan), scientific Latin names, codes, numbers, dates
4. Return ONLY the translated document — no commentary, no explanation, no markdown fences

FORMATTING — THIS IS CRITICAL:
- Return the translated text in EXACTLY the same format as the input
- Keep all table pipes, dashes, rows, columns, blank lines, and spacing identical
- Keep all section numbers and headers in place
- Do not add, remove, or merge any rows, columns, or lines
- Do not wrap the output in code fences or quotes


{page_text}"""


# Pull NON_ENGLISH pages that need translation -Non english ONly

pages_to_translate = s.sql(f"""
    SELECT
        DOC_ID,
        PAGE_INDEX,
        PAGE_NUMBER,
        PAGE_CONTENT
    FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_PAGES
    WHERE LANGUAGE_RESULT          = 'NON_ENGLISH'
      AND PAGE_CONTENT_TRANSLATED  IS NULL
      AND PAGE_CONTENT             IS NOT NULL
      AND TRIM(PAGE_CONTENT)      <> ''
""").collect()

info(f"Pages needing translation: {len(pages_to_translate)}")

translation_results = []
translation_errors  = []

if not pages_to_translate:
    info("Nothing to translate.")


# Translate all NON_ENGLISH pages

if pages_to_translate:
    values_list = ', '.join(
        f"('{r['DOC_ID']}', {r['PAGE_INDEX']})"
        for r in pages_to_translate
    )

    try:
        raw_results = s.sql(f"""
            SELECT
                p.DOC_ID,
                p.PAGE_INDEX,
                p.PAGE_NUMBER,
                p.PAGE_CONTENT,
                LENGTH(p.PAGE_CONTENT)  AS CONTENT_CHARS,
                LENGTH(?)               AS PROMPT_CHARS,
                AI_COMPLETE(
                    '{TRANSLATE_MODEL}',
                    REPLACE(?, '{{page_text}}', p.PAGE_CONTENT)
                ) AS TRANSLATED
            FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_PAGES p
            JOIN (
                SELECT DISTINCT DOC_ID, PAGE_INDEX
                FROM (VALUES {values_list}) AS t(DOC_ID, PAGE_INDEX)
            ) AS to_translate
                ON  p.DOC_ID     = to_translate.DOC_ID
                AND p.PAGE_INDEX = to_translate.PAGE_INDEX
            WHERE p.PAGE_CONTENT IS NOT NULL
              AND TRIM(p.PAGE_CONTENT) <> ''
        """, params=[TRANSLATE_PROMPT, TRANSLATE_PROMPT]).collect()

        info(f"  {len(raw_results)} page(s) returned from Cortex")

    except Exception as e:
        error(f"  SQL translation failed: {e}")
        raw_results = []

    # Parse results
    for row in raw_results:
        doc_id     = row['DOC_ID']
        page_index = row['PAGE_INDEX']
        raw        = row['TRANSLATED']

        try:
            if not raw or not raw.strip():
                raise ValueError("Empty response from AI_COMPLETE")

            # Strip markdown fences if present
            translated = raw.strip()
            if translated.startswith('```'):
                parts      = translated.split('```')
                translated = parts[1].strip()
                if translated.lower().startswith(('text', 'md', 'markdown')):
                    translated = translated.split('\n', 1)[1].strip()
                    
            # Normalize escaped newlines to actual newlines
            translated = translated.replace('\\n', '\n') 

            input_tokens  = (row['PROMPT_CHARS'] + row['CONTENT_CHARS']) // 4
            output_tokens = len(translated) // 4

            translation_results.append({
                'DOC_ID':                  doc_id,
                'PAGE_INDEX':              page_index,
                'PAGE_NUMBER':             row['PAGE_NUMBER'],
                'PAGE_CONTENT_TRANSLATED': translated,
                'INPUT_TOKENS':            input_tokens,
                'OUTPUT_TOKENS':           output_tokens,
            })

            info(f"  [OK] DOC_ID: {doc_id} PAGE: {page_index} - "
                 f"{row['CONTENT_CHARS']:,} chars translated")

        except Exception as e:
            translation_errors.append({
                'doc_id':     doc_id,
                'page_index': page_index,
                'error':      str(e),
            })
            error(f"  [FAIL] DOC_ID: {doc_id} PAGE: {page_index}: {e}")


In [ ]:
if translation_results:
    now = now_ast()

    s.write_pandas(
        pd.DataFrame([{
            'DOC_ID':                  r['DOC_ID'],
            'PAGE_INDEX':              r['PAGE_INDEX'],
            'PAGE_CONTENT_TRANSLATED': r['PAGE_CONTENT_TRANSLATED'],
            'TOKENS_IN':               r['INPUT_TOKENS'],
            'TOKENS_OUT':              r['OUTPUT_TOKENS'],
            'CALLED_AT':               now,
        } for r in translation_results]),
        table_name='TRANSLATION_STAGING',
        database=DB, schema=PROCESSING_SCHEMA,
        overwrite=True,
        auto_create_table=True,
    )

    # Update DOCUMENTS_PAGES 
    s.sql(f"""
        UPDATE {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_PAGES p
        SET p.PAGE_CONTENT_TRANSLATED = t.PAGE_CONTENT_TRANSLATED
        FROM {DB}.{PROCESSING_SCHEMA}.TRANSLATION_STAGING t
        WHERE p.DOC_ID     = t.DOC_ID
          AND p.PAGE_INDEX = t.PAGE_INDEX
    """).collect()

    info(f"Updated {len(translation_results)} page(s) in DOCUMENTS_PAGES")

    # Insert LLM_USAGE from staging
    s.sql(f"""
        INSERT INTO {DB}.{AUDIT_SCHEMA}.LLM_USAGE
            (DOC_ID, CHILD_DOC_ID, PIPELINE_STEP, MODEL_NAME,
             TOKENS_IN, TOKENS_OUT, CALLED_AT)
        SELECT
            DOC_ID,
            NULL,
            'TRANSLATE',
            '{TRANSLATE_MODEL}',
            TOKENS_IN,
            TOKENS_OUT,
            CALLED_AT
        FROM {DB}.{PROCESSING_SCHEMA}.TRANSLATION_STAGING
    """).collect()

    info(f"Wrote {len(translation_results)} LLM_USAGE row(s)")

    # Drop staging
    s.sql(f"""
        DROP TABLE IF EXISTS {DB}.{PROCESSING_SCHEMA}.TRANSLATION_STAGING
    """).collect()

In [ ]:
# Summary
print(f"\n Translation summary")
print(f"  Translated successfully : {len(translation_results)}")
print(f"  Errors                  : {len(translation_errors)}")
if translation_results:
    print(f"  Total input tokens     : "
          f"{sum(r['INPUT_TOKENS'] for r in translation_results):,}")
    print(f"  Total output tokens    : "
          f"{sum(r['OUTPUT_TOKENS'] for r in translation_results):,}")

if translation_errors:
    print("\n  Failed pages:")
    for e in translation_errors:
        print(f"    DOC_ID: {e['doc_id']} PAGE: {e['page_index']}: {e['error']}")

print(f"\n Language result breakdown after translation ")
s.sql(f"""
    SELECT
        LANGUAGE_RESULT,
        COUNT(DISTINCT DOC_ID)              AS DOCS,
        COUNT(*)                            AS PAGES,
        COUNT(PAGE_CONTENT_TRANSLATED)      AS PAGES_WITH_TRANSLATION,
        COUNT(*) - COUNT(PAGE_CONTENT_TRANSLATED)
                                            AS PAGES_WITHOUT_TRANSLATION
    FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_PAGES
    WHERE LANGUAGE_RESULT IS NOT NULL
    GROUP BY LANGUAGE_RESULT
    ORDER BY LANGUAGE_RESULT
""").show()

print(f"\n Sample translated pages")
s.sql(f"""
    SELECT
        DOC_ID,
        PAGE_NUMBER,
        LANGUAGE_RESULT,
        LEFT(PAGE_CONTENT, 150)             AS ORIGINAL_PREVIEW,
        LEFT(PAGE_CONTENT_TRANSLATED, 150)  AS TRANSLATED_PREVIEW
    FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_PAGES
    WHERE LANGUAGE_RESULT         = 'NON_ENGLISH'
      AND PAGE_CONTENT_TRANSLATED IS NOT NULL
    LIMIT 5
""").show()